# 07 · Backtesting

Cost-aware backtest of model signals with equity curve and trade log.


In [ ]:
import sys, warnings
warnings.filterwarnings('ignore')
sys.path.append('..')  # import the src package from the repo root
from src.utils.config import load_config
from src.data import MarketDataLoader
from src.cleaning import DataCleaner
cfg = load_config(env='research')
TICKER = cfg.data.tickers[0]
import numpy as np, pandas as pd
from src.features import FeatureEngineer
from src.models import build_model
from src.backtesting import BacktestEngine
from src.reporting import plot_equity_curve, plot_drawdown
df = DataCleaner(cfg).clean(MarketDataLoader(cfg).load_one(TICKER))
X, y = FeatureEngineer(cfg).build_dataset(df)
split = int(len(X)*0.7)
model = build_model('xgboost', cfg).fit(X[:split], y[:split])
preds = model.predict(X[split:])
prices = df['close'].reindex(X.index)[split:]
pos = pd.Series(np.sign(preds[-len(prices):]), index=prices.index)
res = BacktestEngine(cfg).run(prices, pos)
res.metrics


In [ ]:
res.trades.head()


In [ ]:
plot_equity_curve(res.equity_curve, benchmark=prices, out='../reports/equity.png')
plot_drawdown(res.equity_curve, out='../reports/drawdown.png')
